In [57]:
import pandas as pd
import numpy as np
from google import genai
from tqdm import tqdm
import faiss
from dotenv import load_dotenv
import os
import time

In [59]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
EMBED_MODEL = os.getenv("EMBED_MODEL")
GEN_MODEL = os.getenv("GEN_MODEL")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
df = pd.read_csv("../data/movie_infos.csv")

In [10]:
def convert_prompt(row):
    title = row["title"]
    genres = row["genres"]
    tags = row["tags"]
    # link = f"https://www.imdb.com/title/{row["imdbId"]}/"
    # rating = round(row["weight_rating"],4)

    return f"Movie's title: {title}\ngenres: {genres}\ntags: {tags}"


df["page_content"] = df.apply(convert_prompt, axis=1)

In [11]:
df

,movieId,title,genres,tags,year,rating,imdbId,page_content
0,1,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy","adventure,animated,animation,cartoon,cgi,child...",1995.0,3.921240,tt0114709,Movie's title: Toy Story (1995)\ngenres: Adven...
1,2,Jumanji (1995),"Adventure,Children,Fantasy","adventure,animals,big budget,childhood,childre...",1995.0,3.211977,tt0113497,Movie's title: Jumanji (1995)\ngenres: Adventu...
2,3,Grumpier Old Men (1995),"Comedy,Romance","comedy,good sequel,original,sequel,sequels",1995.0,3.151040,tt0113228,Movie's title: Grumpier Old Men (1995)\ngenres...
3,4,Waiting to Exhale (1995),"Comedy,Drama,Romance","chick flick,girlie movie,romantic,unlikely fri...",1995.0,2.861393,tt0114885,Movie's title: Waiting to Exhale (1995)\ngenre...
4,5,Father of the Bride Part II (1995),Comedy,"comedy,destiny,family,father daughter relation...",1995.0,3.064592,tt0113041,Movie's title: Father of the Bride Part II (19...
...,...,...,...,...,...,...,...,...
10337,130578,The Gunman (2015),"Action,Thriller","action,assassin,assassination,good action,real...",2015.0,3.000000,tt2515034,Movie's title: The Gunman (2015)\ngenres: Acti...
10338,130840,Spring (2015),"Horror,Romance,Sci-Fi","cinematography,creepy,horror,immortality,love ...",2015.0,3.500000,tt3395184,"Movie's title: Spring (2015)\ngenres: Horror,R..."
10339,131013,Get Hard (2015),Comedy,"buddy movie,coen bros,comedy,crude humor,foul ...",2015.0,2.500000,tt2561572,Movie's title: Get Hard (2015)\ngenres: Comedy...
10340,131168,Phoenix (2014),Drama,"betrayal,camp,cinematography,criterion,dramati...",2014.0,3.500000,tt2764784,Movie's title: Phoenix (2014)\ngenres: Drama\n...


In [99]:
OUTPUT_DIMENSION = 768
index = faiss.IndexFlatIP(OUTPUT_DIMENSION)

In [ ]:
# index.add(np.array([[1, 2., 3.], [2,3,4]]))

In [104]:
index.ntotal

10342

In [101]:
def embed_contents(batch):
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=batch,
        config={
            "output_dimensionality": OUTPUT_DIMENSION
        }
    )

    embeddings = np.array([vector.values for vector in res.embeddings])

    # normalize to work with cosine similarity
    normalized_vectors = embeddings / \
        np.linalg.norm(embeddings, axis=1, keepdims=True)

    return normalized_vectors

In [102]:
batch_size = 100
for i in tqdm(range(0, df.shape[0], batch_size)):
    batch = df["page_content"][i:i+batch_size].tolist()
    # print(batch)
    embed_vectors = embed_contents(batch)
    index.add(embed_vectors)

100%|██████████| 104/104 [04:17<00:00,  2.47s/it]


In [103]:
output_file = "../output_index/text-embed-04"
faiss.write_index(index, output_file)